# Real Sentinel-2 inspection
Executed against the fixed 2024 study. No accuracy claims.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio

root = Path.cwd() if Path("README.md").exists() else Path.cwd().parent
manifest = json.loads((root / "reports/run-manifest.json").read_text())
manifest["grid"], manifest["common_pixels"], manifest["common_fraction"]

({'crs': 'EPSG:32615',
  'resolution_m': 20,
  'width': 172,
  'height': 226,
  'transform': [20.0, 0.0, 374600.0, 0.0, -20.0, 4588700.0, 0.0, 0.0, 1.0]},
 38720,
 0.9960897303972011)

In [2]:
table = pd.read_csv(root / "reports/time-series.csv")
table[["date", "NDVI_median", "NDMI_median", "valid_fraction"]]

,date,NDVI_median,NDMI_median,valid_fraction
0,2024-04-15,0.383223,-0.088889,0.997582
1,2024-05-25,0.570682,0.044812,0.997993
2,2024-06-09,0.582466,0.034859,0.998714
3,2024-07-24,0.748399,0.269184,0.999743
4,2024-08-31,0.778708,0.245941,0.998662
5,2024-09-30,0.504476,-0.062554,1.000000
6,2024-10-05,0.450336,-0.073114,1.000000


In [3]:
assert len(table) == 7
assert table.common_pixels.nunique() == 1
with rasterio.open(sorted((root / "data/processed").glob("*.tif"))[0]) as ds:
    print(ds.crs, ds.res, ds.descriptions, ds.nodata)
    assert ds.count == 4 and ds.res == (20, 20)
    assert str(ds.crs) == "EPSG:32615"

EPSG:32615 (20.0, 20.0) ('NDVI', 'NDWI', 'NDMI', 'SAVI') -9999.0


In [4]:
from satellite.indices import compute_indices

bands = {
    "red": np.array([[0.1]]),
    "green": np.array([[0.15]]),
    "nir": np.array([[0.5]]),
    "swir16": np.array([[0.25]]),
}
compute_indices(bands, np.array([[True]]))

{'NDVI': array([[0.6666667]], dtype=float32),
 'NDWI': array([[-0.53846157]], dtype=float32),
 'NDMI': array([[0.33333334]], dtype=float32),
 'SAVI': array([[0.54545456]], dtype=float32)}

The single pixel above is an explicitly artificial arithmetic example, not source imagery. The study statistics and maps use real public COG data. Spatial variation is not predictive uncertainty; no crop identity or physiological stress has been validated.